# Multithreading in Python — The `threading` Module, Synchronization & I/O Concurrency

> **Topic:** Multithreading & Thread Synchronization | **Folder:** Concurrency

**Multithreading** allows multiple threads of execution to run within a **single process**, sharing
the same memory address space.

However, because of Python's **Global Interpreter Lock (GIL)**, threads **cannot run in parallel on different CPU cores**.
This makes multithreading highly suitable for **I/O-bound tasks** (e.g., network API requests, disk file access, database queries)
where threads spend most of their time waiting for external responses, but **not for computational (CPU-bound) tasks**.

---

## Table of Contents
1. [Multithreading Fundamentals & Memory Space](#1.-Multithreading-Fundamentals-&-Memory-Space)
2. [The `threading.Thread` API & Daemon Threads](#2.-The-`threading.Thread`-API-&-Daemon-Threads)
3. [Thread Pools (`ThreadPoolExecutor`)](#3.-Thread-Pools-(ThreadPoolExecutor))
4. [Race Conditions & Shared Memory Mutation](#4.-Race-Conditions-&-Shared-Memory-Mutation)
5. [Thread Synchronization Primitives (`Lock`, `RLock`, `Semaphore`, `Event`)](#5.-Thread-Synchronization-Primitives-(Lock,-RLock,-Semaphore,-Event))
6. [Thread-Safe Queues (`queue.Queue`) & Producer-Consumer Pattern](#6.-Thread-Safe-Queues-(queue.Queue)-&-Producer-Consumer-Pattern)
7. [Thread-Local Data (`threading.local`)](#7.-Thread-Local-Data-(threading.local))
8. [Empirical Benchmark: I/O-Bound vs. CPU-Bound Multithreading](#8.-Empirical-Benchmark:-I/O-Bound-vs.-CPU-Bound-Multithreading)
9. [Quick Reference Card](#9.-Quick-Reference-Card)


---
## 1. Multithreading Fundamentals & Memory Space

Unlike processes (which have completely separate isolated memory spaces), threads share global variables, heap memory,
and file descriptors within the same parent process.

| Paradigm | Memory Sharing | Creation Overhead | Best Use Case |
|----------|----------------|-------------------|---------------|
| **Multithreading (`threading`)** | Shared memory space | Light ($\sim 8$KB stack) | Network calls, disk I/O, user interfaces |
| **Multiprocessing (`multiprocessing`)** | Isolated memory space | Heavy (Full Python process) | CPU-bound math, image processing |


---
## 2. The `threading.Thread` API & Daemon Threads

- `Thread(target, args)`: Spawns a new native thread.
- `daemon=True`: Daemon threads run silently in the background and are automatically terminated when the main thread exits.


In [ ]:
import threading
import time

def print_numbers(thread_name, count):
    for i in range(1, count + 1):
        time.sleep(0.05)  # Simulating minor I/O latency
        print(f"  [{thread_name}] Step {i}/{count}")

# Spawning threads
t1 = threading.Thread(target=print_numbers, args=("Thread-A", 3))
t2 = threading.Thread(target=print_numbers, args=("Thread-B", 3))

t1.start(); t2.start()
t1.join(); t2.join()  # Wait for threads to finish
print("Both threads completed.")


---
## 3. Thread Pools (`ThreadPoolExecutor`)

`concurrent.futures.ThreadPoolExecutor` simplifies thread management by using a pool of reusable worker threads.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_simulation(url_id):
    time.sleep(0.1)
    return f"Content of URL_{url_id}"

urls = [101, 102, 103, 104, 105]

with ThreadPoolExecutor(max_workers=3) as executor:
    future_map = {executor.submit(download_simulation, url): url for url in urls}
    for future in as_completed(future_map):
        url_id = future_map[future]
        print(f"Downloaded {url_id} -> {future.result()}")


---
## 4. Race Conditions & Shared Memory Mutation

Because memory is shared, un-synchronized concurrent updates to a shared variable can cause **race conditions**!


In [ ]:
shared_counter = 0

def unsafe_increment():
    global shared_counter
    for _ in range(100_000):
        shared_counter += 1  # Not atomic! Read -> Modify -> Write

# Spawning 2 threads modifying shared_counter
t1 = threading.Thread(target=unsafe_increment)
t2 = threading.Thread(target=unsafe_increment)
t1.start(); t2.start(); t1.join(); t2.join()

print(f"Expected Counter: 200000")
print(f"Actual Counter  : {shared_counter} (Race condition check)")


---
## 5. Thread Synchronization Primitives (`Lock`, `RLock`, `Semaphore`, `Event`)

- **`Lock`**: Guarantees mutual exclusion (`with lock:`).
- **`Semaphore`**: Limits max active concurrent threads.
- **`Event`**: Thread signaling flag (`set()`, `wait()`).


In [ ]:
safe_counter = 0
counter_lock = threading.Lock()

def safe_increment():
    global safe_counter
    for _ in range(50_000):
        with counter_lock:  # Mutex protection
            safe_counter += 1

t1 = threading.Thread(target=safe_increment)
t2 = threading.Thread(target=safe_increment)
t1.start(); t2.start(); t1.join(); t2.join()

print(f"Safe Counter with Lock: {safe_counter}")


---
## 6. Thread-Safe Queues (`queue.Queue`) & Producer-Consumer Pattern


In [ ]:
import queue

work_queue = queue.Queue()

def worker():
    while True:
        item = work_queue.get()
        if item is None: break
        print(f"  [Queue Worker] Processed item: {item}")
        work_queue.task_done()

worker_thread = threading.Thread(target=worker, daemon=True)
worker_thread.start()

# Produce work items
for i in ["Task 1", "Task 2", "Task 3"]:
    work_queue.put(i)

work_queue.join()  # Wait for all tasks to be marked done
work_queue.put(None)  # Sentinel to stop worker


---
## 7. Thread-Local Data (`threading.local`)

Maintains independent isolated state per thread without global collision.


In [ ]:
thread_local = threading.local()

def process_user(user_name):
    thread_local.user = user_name
    time.sleep(0.05)
    print(f"  Thread {threading.current_thread().name} is handling user: {thread_local.user}")

t1 = threading.Thread(target=process_user, args=("Alice",), name="T1")
t2 = threading.Thread(target=process_user, args=("Bob",), name="T2")
t1.start(); t2.start(); t1.join(); t2.join()


---
## 8. Empirical Benchmark: I/O-Bound vs. CPU-Bound Multithreading


In [ ]:
# Comparing 10 I/O tasks vs 10 CPU tasks in Multithreading
def io_task(delay=0.1):
    time.sleep(delay)  # GIL released during sleep

def cpu_task(n=2_000_000):
    s = 0
    for i in range(n): s += i
    return s

# --- 1. I/O-BOUND BENCHMARK ---
t0 = time.perf_counter()
for _ in range(10): io_task(0.1)
t_io_seq = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as ex:
    list(ex.map(io_task, [0.1]*10))
t_io_threads = time.perf_counter() - t0

print(f"I/O Tasks Sequential Time: {t_io_seq:.4f}s")
print(f"I/O Tasks Threaded Time  : {t_io_threads:.4f}s (~{t_io_seq/t_io_threads:.1f}x SPEEDUP!)")

# --- 2. CPU-BOUND BENCHMARK ---
t0 = time.perf_counter()
for _ in range(4): cpu_task()
t_cpu_seq = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(cpu_task, [2_000_000]*4))
t_cpu_threads = time.perf_counter() - t0

print(f"\nCPU Tasks Sequential Time: {t_cpu_seq:.4f}s")
print(f"CPU Tasks Threaded Time  : {t_cpu_threads:.4f}s (NO speedup due to GIL!)")


---
## 9. Quick Reference Card


In [ ]:
# ==================================================================
# MULTITHREADING – QUICK REFERENCE
# ==================================================================
from concurrent.futures import ThreadPoolExecutor
import threading

# ThreadPoolExecutor (Preferred API for I/O):
# with ThreadPoolExecutor(max_workers=10) as ex:
#     results = list(ex.map(fetch_fn, urls))

# Lock Protection:
# lock = threading.Lock()
# with lock: shared_var += 1


---
## Summary

| Component / API | Time / Memory Overhead | Primary Use Case |
|-----------------|------------------------|------------------|
| **`ThreadPoolExecutor`** | Light ($\sim 8$KB stack/thread) | Concurrent I/O calls (HTTP requests, file access) |
| **`threading.Lock`** | Very low | Preventing race conditions on shared memory |
| **`queue.Queue`** | Thread-safe queue | Producer-Consumer task passing |
| **`threading.local`** | Isolated dict per thread | Storing thread-specific session/connection objects |

---
*Next up: **Object-Oriented Programming (OOP)***
